# Middle Mile Fleet Data Extraction

**Objective:** Automate middle fleet capacity utilization data preprocessing of monthly delivery data into a format suitable for operational analysis.

**Key Components:**
1. **Extract:** raw CSV delivery data from the Colab upload environment.
2. **Transform:**
    - parse delivery dates
    - normalize time slot labels
    - add week/month metadata
    - sort by hub, delivery date, and time slot
    - Aggregate the transformed data into a pivoted summary by delivery date, hub, week, month, and time slot.
3. **Load:** Export the final results to a CSV/Excel-friendly format for operational analysis.

# Setup, Imports, and Configuration

In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz
from google.colab import files
import os
import gspread
from google.colab import auth
from google.auth import default

LOCAL_TZ = pytz.timezone('Asia/Jakarta')
DATE_FORMAT = "%m-%d-%Y" 
CURRENT_DATE_STR = datetime.now(LOCAL_TZ).strftime(DATE_FORMAT)

UPLOAD_DIR = '/content/'

TARGET_COLUMS = [
    'delivery_date',  'hubs', 'time_slot', 'total_weight_perorder', 'order_no'
]


# 1. Extract
Upload the raw CSV file into the Colab environment and load it into a Pandas DataFrame.

In [ ]:
def extract_data(upload_dir: str) -> pd.DataFrame:
    print("Please upload the daily delivery CSV file:")
    uploaded = files.upload()
    
    csv_files = [f for f in os.listdir(upload_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the upload directory.")
        
    file_path = os.path.join(upload_dir, csv_files[0])
    df = pd.read_csv(file_path)
    print(f"Successfully extracted {len(df)} rows from {csv_files[0]}")
    return df

# Execute Extract
raw_df = extract_data(UPLOAD_DIR)

# 2. Transform
Clean data types, filter the necessary columns, and sort the data for operational efficiency.

In [ ]:
def clean_and_transform(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Applies type casting, column filtering, and sorting."""
    # Always operate on a copy to preserve raw data
    transformed_df = df.copy()
    
    # Convert 'delivery_date' to datetime objects
    transformed_df['delivery_date'] = pd.to_datetime(transformed_df['delivery_date'], errors='coerce')
    
    # Add 'week' and 'month' columns
    transformed_df.loc[:, 'week'] = transformed_df['delivery_date'].dt.isocalendar().week
    transformed_df.loc[:, 'month'] = transformed_df['delivery_date'].dt.month

    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-12bb', 'slot-1')
    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-1bb', 'slot-1')
    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-0bb', 'slot-0')
    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-b2b-2', 'slot-0')
    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-sameday03', 'slot-2')
    transformed_df.loc[:, 'time_slot'] = transformed_df['time_slot'].str.replace('slot-13', 'slot-sameday')

    # Sort by multiple columns:
    sorted_df = transformed_df.sort_values(by=['hubs', 'delivery_date', 'time_slot'], ascending=[True, True,True])
    
    return transformed_df

In [ ]:
# Execute Transform
clean_df = clean_and_transform(raw_df, columns=TARGET_COLUMS)

In [ ]:
# Create the pivot table
pivot_table = clean_df.pivot_table(
    values=['total_weight_perorder', 'order_no'], 
    index=['delivery_date', 'hubs', 'week', 'month'], 
    columns='time_slot', 
    aggfunc={'total_weight_perorder': 'sum', 'order_no': 'count'}
    )

# Reset the index to make 'week' and 'month' regular columns
pivot_table_reset = pivot_table.reset_index()

# Flatten the MultiIndex columns more carefully
new_column_names = []
for col in pivot_table_reset.columns:
    if isinstance(col, tuple):
        # Join non-empty parts of the tuple with an underscore
        # This handles cases like ('delivery_date', '') becoming 'delivery_date'
        cleaned_name = '_'.join(filter(None, col))
        new_column_names.append(cleaned_name)
    else:
        new_column_names.append(col) # Keep single-level columns as is

pivot_table_reset.columns = new_column_names

# Get the list of columns
cols = pivot_table_reset.columns.tolist()

# Identify the columns to move (these should now be 'week' and 'month' without underscores)
cols_to_move = ['week', 'month']
cols_to_keep = [col for col in cols if col not in cols_to_move]

# Create the new ordered list of columns
new_order = cols_to_keep + cols_to_move

# Reindex the DataFrame with the new column order
pivot_table_ordered = pivot_table_reset[new_order]

# Set the index back to delivery_date and hubs (using original names)
pivot_table = pivot_table_ordered.set_index(['delivery_date', 'hubs'])

# Print the pivot table
display(pivot_table.head(3))

In [ ]:
pivot_table.to_csv('fleet_capacity_utilization.csv', index=True, header=False)

# 3. Load
Load dataframe into existing Google Sheets

In [ ]:
# 1. Authenticate Google account
# (A pop-up window will ask for permission to access Google Drive/Sheets)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Load CSV file using Pandas
csv_filename = 'fleet_capacity_utilization.csv'  # e.g., 'sales_data.csv'
df = pd.read_csv(csv_filename)

# Clean up any NaN (empty) values so the Google Sheets API doesn't throw an error
df = df.fillna('')

# Convert the Pandas DataFrame into a list of lists (the format Google Sheets expects)
data_to_append = df.values.tolist()

# 3. Open existing Google Sheet
# You can use the full URL of Google Sheet here
sheet_url = 'https://docs.google.com/spreadsheets/d/1LbsjF_smDkSw8Cx_WmHNX3I5xDMqk4J4fgYcOj7dxW4/edit?gid=1632138732#gid=1632138732' 
spreadsheet = gc.open_by_url(sheet_url)
worksheet = spreadsheet.get_worksheet(0) 

# 4. Append the data to the bottom of the sheet
worksheet.append_rows(data_to_append)

print(f"Successfully appended {len(data_to_append)} rows to Google Sheet!")